# Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.0 MB/s eta 0:00:00


# Load Dataset

In [ ]:
# from google.colab import files
# uploaded = files.upload()

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

filepath = "/content/drive/MyDrive/amazon_reviews_us_Electronics_v1_00.tsv"

use_cols = [
    "product_id",
    "product_title",
    "review_body",
    "star_rating",
    "helpful_votes",
    "verified_purchase"
]

chunk_size = 100_000
chunks = []

for chunk in pd.read_csv(
    filepath,
    sep="\t",
    usecols=use_cols,
    on_bad_lines="skip",
    chunksize=chunk_size
):
    chunk = chunk.dropna(subset=["product_id", "product_title", "review_body"])
    chunk["review_length"] = chunk["review_body"].str.split().str.len()
    chunk = chunk[chunk["review_length"] >= 20]  # dropping reviews with less than 20 char
    chunk["verified_purchase"] = (chunk["verified_purchase"] == "Y").astype(int)
    chunks.append(chunk)

    if len(chunks) == 5:   # 🔴 LIMIT DATA (≈500k rows)
        break

df = pd.concat(chunks, ignore_index=True)
print("Loaded rows:", df.shape)


Loaded rows: (227162, 7)


# Data Cleaning

# Keeping 1 reveiw per product

In [ ]:
df = df.sort_values(
    by=["product_id", "verified_purchase", "helpful_votes", "review_length"],
    ascending=[True, False, False, False]
)

df = df.groupby("product_id").first().reset_index()

print("Unique products:", df.shape[0])


Unique products: 40585


# Embedding

In [ ]:
df["embed_text"] = (
    df["product_title"] + ". " +
    df["review_body"]
)

## Loading Embedding On T4 GPU

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Load model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import faiss
import numpy as np
from tqdm import tqdm

In [ ]:
texts = df["embed_text"].tolist()

BATCH_SIZE = 512    # doing in batches because it will take too much time and computational power to load all the data at once
embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings.append(emb)

embeddings = np.vstack(embeddings)

print(embeddings.shape)


100%|██████████| 80/80 [01:07<00:00,  1.18it/s]

(40585, 384)


In [ ]:
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS vectors:", index.ntotal)

FAISS vectors: 40585


In [ ]:
faiss_path = "/content/drive/MyDrive/faiss_phase1.index"
faiss.write_index(index, faiss_path)

df_path = "/content/drive/MyDrive/product_df_phase1.pkl"
df.to_pickle(df_path)

print("Saved FAISS index to:", faiss_path)
print("Saved product metadata to:", product_df_path)


Saved FAISS index to: /content/drive/MyDrive/faiss_phase1.index
Saved product metadata to: /content/drive/MyDrive/product_df_phase1.pkl


Testing Recommendation System

In [ ]:
def recommend_similar_products(query, top_k=5):
    q_emb = model.encode([query])
    faiss.normalize_L2(q_emb)

    scores, indices = index.search(q_emb, top_k)

    results = df.iloc[indices[0]][
        ["product_id", "product_title"]
    ].copy()

    results["similarity"] = scores[0]
    return results


In [ ]:
recommend_similar_products("wireless noise cancelling headphones")

,product_id,product_title,similarity
32703,B00OLZCYLI,Beats Studio Wireless Over-Ear Headphone,0.708117
38913,B00X9YQ0YE,KINYO PW8899 Over the Ear Wireless Headphones,0.690948
38322,B00W4WVU38,Uniden Passive Noise Cancelling On-Ear Stereo ...,0.689561
10833,B005FVDTAW,Sony MDRNC200D Digital Noise-Canceling Headpho...,0.683586
9484,B004P7O26W,Sony MDRNC13 Noise-Canceling Headphones,0.682536


# Phase 2 (LLM + RAG)

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.6 MB/s eta 0:00:00


loading cleaned DF that is alligned with cosing embedding

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_pickle("/content/drive/MyDrive/df_phase1.pkl")
print(df.shape)

(40585, 8)


Loading embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading and comparing Vector DB and DF

In [ ]:
import faiss

faiss_path = "/content/drive/MyDrive/faiss_phase1.index"
index = faiss.read_index(faiss_path)

print("FAISS vectors:", index.ntotal)
assert index.ntotal == len(df), "FAISS index does not match df rows!"

FAISS vectors: 40585


Retrieving best products according to search

In [ ]:
def retrieve_products(query, top_k=5):
    # embed query
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    # search FAISS
    distances, indices = index.search(q_emb, top_k)

    results = df.iloc[indices[0]].copy()
    results["distance"] = distances[0]

    return results

Test Retrieval

In [ ]:
retrieve_products("wireless noise cancelling headphones", top_k=5)

,product_id,product_title,star_rating,helpful_votes,verified_purchase,review_body,review_length,embed_text,distance
32703,B00OLZCYLI,Beats Studio Wireless Over-Ear Headphone,3,5,1,"The battery doesn't last too long, and compare...",180,Beats Studio Wireless Over-Ear Headphone. The ...,0.708117
38913,B00X9YQ0YE,KINYO PW8899 Over the Ear Wireless Headphones,2,0,1,16.00 it does the job; at times it does have n...,29,KINYO PW8899 Over the Ear Wireless Headphones....,0.690948
38322,B00W4WVU38,Uniden Passive Noise Cancelling On-Ear Stereo ...,1,1,1,Wishing negative stars option for these. No no...,42,Uniden Passive Noise Cancelling On-Ear Stereo ...,0.689561
10833,B005FVDTAW,Sony MDRNC200D Digital Noise-Canceling Headpho...,4,0,1,The noise-cancellation feature works very well...,135,Sony MDRNC200D Digital Noise-Canceling Headpho...,0.683586
9484,B004P7O26W,Sony MDRNC13 Noise-Canceling Headphones,5,2,1,I had a pair of these that I used for about 3 ...,121,Sony MDRNC13 Noise-Canceling Headphones. I had...,0.682536


LLM setup (Gemini Flash 2.5 Flash Setup)

In [ ]:
!pip install --upgrade google-genai

In [ ]:
from google import genai

client = genai.Client(api_key="XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


Building Promt for LLM input From RAG

In [ ]:
import re

def build_prompt(query, retrieved_products):
    """
    Build the RAG prompt for Gemini 2.5 Flash.

    Args:
        query (str): User query.
        retrieved_products (pd.DataFrame): Top-k products from FAISS.

    Returns:
        str: Fully formatted prompt for LLM.
    """
    # Detect price keywords
    price_terms = ["$", "under", "less than", "max", "price"]
    mentions_price = any(term in query.lower() for term in price_terms)
    price_note = ""
    if mentions_price:
        price_note = "Note: Price filtering is not available. Recommendations are based on reviews and ratings only.\n\n"

    # Build context
    context = ""
    for _, row in retrieved_products.iterrows():
        context += f"""
Product Title: {row['product_title']}
Star Rating: {row['star_rating']}
Review: {row['review_body']}
---
"""

    # Construct final prompt
    prompt = f"""
You are a product recommendation assistant.

Rules:
- Use ONLY the provided products.
- Do NOT use outside knowledge.
- Recommend the best option and explain why.

{price_note}

User Query:
{query}

Products:
{context}

Answer:
"""
    return prompt


# Recommendation

In [ ]:
import pandas as pd

def recommend_with_llm(query, top_k=5):
    """
    Retrieve top_k products from FAISS, filter by user-specified rating,
    then generate recommendation using Gemini 2.5 Flash.
    """
    # 1️⃣ Retrieve top_k products from FAISS
    retrieved = retrieve_products(query, top_k=top_k)  # make sure this returns DataFrame

    # 2️⃣ Extract minimum rating from query
    rating_match = re.search(r"(\d(?:\.\d)?)\s*star", query.lower())
    min_rating = float(rating_match.group(1)) if rating_match else 0
    retrieved = retrieved[retrieved["star_rating"] >= min_rating]

    if retrieved.empty:
        return "No products match the required rating criteria."

    # 3️⃣ Build prompt
    prompt = build_prompt(query, retrieved)

    # 4️⃣ Generate response using Gemini 2.5 Flash
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text


In [ ]:
query = "suggest a smart watch with good battery life and bluetooth calling feature"
print(recommend_with_llm(query, top_k=5))


NameError: name 'retrieve_products' is not defined